<!-- CURRICULUM_HEADER_START -->
<div class="mh"><p class="mh-crumb"><a href="/series/prediction/index.html">Weather, Climate and Prediction</a><span class="sep">·</span>Part 3 · Simple models that work</p><p class="mh-facts"><span class="lvl lvl-introductory">Introductory</span></p><p class="mh-assumes">Assumes <a href="/blogs/climate-vs-weather/climate-vs-weather.html">Climate vs weather</a> and <a href="/blogs/rainfall-predictability/why-cant-we-forecast-rain-six-months-out.html">Why can&#x27;t we forecast rain six months out?</a>.</p></div>
<!-- CURRICULUM_HEADER_END -->

::: {.callout-note appearance="simple" icon=false}
## TL;DR

A first-order Markov chain forecasts tomorrow's wet-or-dry state from today's alone. Fitted to four years of Seattle daily rainfall, the two-state chain reproduces the observed wet-day fraction and holds up when today is wet, but yesterday still matters after a dry day. Afterwards you will be able to build a transition matrix from daily data, check its stationary distribution, and test the memoryless assumption directly.
:::

The previous module showed that a specific day's rainfall is unknowable six months out: the atmosphere is chaotic and small uncertainties snowball within about two weeks. So here is a much smaller question: **if it is raining right now, what is the chance it is still raining tomorrow?**

For that question meteorologists reach for one of the simplest tools available, a **Markov chain**, which assumes tomorrow depends only on today, not on the history that led here. That sounds too lazy to work. We test it on four years of real daily weather data.


## What a Markov chain is

A Markov chain hops between a fixed set of **states**, and the probability of the next state depends only on the **current** state, not on how it got there. This is the **Markov property**, or "memorylessness".

Think of a board-game token: to decide where it moves next you only need its current square and a die roll, not the path it took. Weather, modelled this way, has two squares: **Wet** and **Dry**.

It is a bold simplification. Real weather does have memory; a slow-moving storm system does not reset every 24 hours. The question is whether that memory matters enough to model, or whether "today" alone captures most of what we need.


<pre class="mermaid">
stateDiagram-v2
    [*] --> Dry
    Dry --> Dry: stays dry
    Dry --> Wet: turns wet
    Wet --> Wet: stays wet
    Wet --> Dry: turns dry
</pre>

Two states, four possible transitions: that is the whole model. Let's replace the generic arrows with real probabilities.


In [1]:
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display


def show(fig, div_id, height=420):
    """Render a self-contained, interactive Plotly figure (hover, zoom, pan)."""
    fig.update_layout(
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(t=60, b=40, l=50, r=20),
    )
    html = fig.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        default_height=f"{height}px",
        div_id=div_id,
    )
    # Strip the auto-generated SRI hash on the CDN <script> tag -- it's a long
    # base64 string that trips secret scanners as a false positive.
    html = re.sub(r'\s+integrity="[^"]*"\s+crossorigin="anonymous"', "", html)
    display(HTML(html))

## Fitting the chain to Seattle

### The data

Four full years (2012–2015) of daily weather observations for Seattle, Washington: 1,461 days of precipitation, temperature and wind, published in the open-source [vega-datasets](https://github.com/vega/vega-datasets) collection (originally from NOAA). We need one column, daily precipitation, collapsed into a binary **Wet** (precipitation > 0 mm) / **Dry** (no precipitation) label for each day.


In [2]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/vega/vega-datasets/main/data/seattle-weather.csv"
)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df["wet"] = (df["precipitation"] > 0).astype(int)  # 1 = wet day, 0 = dry day

STATE_NAMES = ["Dry", "Wet"]
df[["date", "precipitation", "wet"]].head()

,date,precipitation,wet
0,2012-01-01,0.0,0
1,2012-01-02,10.9,1
2,2012-01-03,0.8,1
3,2012-01-04,20.3,1
4,2012-01-05,1.3,1


### Building the transition matrix

For every pair of consecutive days we ask: what state was it *today*, and what did it become *tomorrow*? Counting all 1,460 transitions and normalising each row gives the chain's **transition matrix**, the probability of moving from any state to any other.


In [3]:
states = df["wet"].values
today, tomorrow = states[:-1], states[1:]

counts = np.zeros((2, 2))
for s0, s1 in zip(today, tomorrow):
    counts[s0, s1] += 1
trans_prob = counts / counts.sum(axis=1, keepdims=True)

pd.DataFrame(
    trans_prob,
    index=[f"today: {s}" for s in STATE_NAMES],
    columns=[f"tomorrow: {s}" for s in STATE_NAMES],
).round(3)

,tomorrow: Dry,tomorrow: Wet
today: Dry,0.756,0.244
today: Wet,0.327,0.673


In [4]:
fig = go.Figure(
    data=go.Heatmap(
        z=trans_prob.tolist(),
        x=STATE_NAMES,
        y=STATE_NAMES,
        colorscale="Blues",
        zmin=0,
        zmax=1,
        text=[[f"{v:.0%}" for v in row] for row in trans_prob],
        texttemplate="%{text}",
        textfont=dict(size=20),
        hovertemplate="today: %{y}<br>tomorrow: %{x}<br>probability: %{z:.1%}<extra></extra>",
        showscale=False,
    )
)
fig.update_layout(
    title="Real transition matrix, fit on 4 years of Seattle weather",
    xaxis_title="Tomorrow",
    yaxis_title="Today",
)
fig.update_yaxes(autorange="reversed")
show(fig, "transition-heatmap", height=380)

### Reading the matrix

A dry day is followed by another dry day about 76% of the time; a wet day by another wet day about 67% of the time. Both states are "sticky", but dry spells are slightly more self-reinforcing than wet ones, at least in Seattle.

The same information as a flow: out of every 1,000 days in each state, where does the chain send them tomorrow?


In [5]:
fig = go.Figure(
    data=go.Sankey(
        node=dict(
            label=["Dry (today)", "Wet (today)", "Dry (tomorrow)", "Wet (tomorrow)"],
            color=["#c98a2b", "#1b6ca8", "#c98a2b", "#1b6ca8"],
            pad=25,
            thickness=18,
        ),
        link=dict(
            source=[0, 0, 1, 1],
            target=[2, 3, 2, 3],
            value=(trans_prob.flatten() * 1000).tolist(),
            color=[
                "rgba(201,138,43,0.4)",
                "rgba(201,138,43,0.4)",
                "rgba(27,108,168,0.4)",
                "rgba(27,108,168,0.4)",
            ],
            hovertemplate="%{value:.0f} of every 1000 days<extra></extra>",
        ),
    )
)
fig.update_layout(
    title="Where does today's weather send tomorrow's? (per 1,000 days)", font_size=12
)
show(fig, "transition-sankey", height=380)

### Does the chain agree with itself?

If we let this chain run forever, the fraction of days in each state should settle into a fixed **stationary distribution**, and that should match the observed fraction of wet and dry days. If it did not, something would be wrong with the model.


In [6]:
eigvals, eigvecs = np.linalg.eig(trans_prob.T)
stationary = eigvecs[:, np.isclose(eigvals, 1)].flatten().real
stationary = stationary / stationary.sum()

observed = np.array([1 - states.mean(), states.mean()])

comparison = pd.DataFrame(
    {
        "Chain's stationary distribution": stationary,
        "Actually observed in the data": observed,
    },
    index=STATE_NAMES,
).round(3)
comparison

,Chain's stationary distribution,Actually observed in the data
Dry,0.573,0.574
Wet,0.427,0.426


They match almost exactly: the chain's long-run behaviour is consistent with Seattle's real climate. That is reassuring but weak, because any correctly fitted Markov chain passes it by construction. The real test is whether the *memoryless* assumption itself holds.


## Does yesterday actually matter?

The Markov property says $P(\text{tomorrow} \mid \text{today}) = P(\text{tomorrow} \mid \text{today}, \text{yesterday}, \ldots)$: knowing more history should not change the forecast. We can check that directly by splitting the data by *both* today's and yesterday's state and seeing whether the forecast for tomorrow shifts.


In [7]:
yesterday, today2, tomorrow2 = states[:-2], states[1:-1], states[2:]

rows = []
for today_state, today_label in enumerate(STATE_NAMES):
    p_markov = tomorrow2[today2 == today_state].mean()
    for yday_state, yday_label in enumerate(STATE_NAMES):
        mask = (today2 == today_state) & (yesterday == yday_state)
        p_full_history = tomorrow2[mask].mean()
        rows.append(
            {
                "today": today_label,
                "yesterday": yday_label,
                "n_days": int(mask.sum()),
                "P(wet tomorrow | today only)": p_markov,
                "P(wet tomorrow | today AND yesterday)": p_full_history,
            }
        )
memory_test = pd.DataFrame(rows)
memory_test.round(3)

,today,yesterday,n_days,P(wet tomorrow | today only),P(wet tomorrow | today AND yesterday)
0,Dry,Dry,632,0.243,0.187
1,Dry,Wet,204,0.243,0.417
2,Wet,Dry,204,0.673,0.657
3,Wet,Wet,419,0.673,0.680


In [8]:
fig = go.Figure()
for today_state, today_label in enumerate(STATE_NAMES):
    subset = memory_test[memory_test["today"] == today_label]
    fig.add_trace(
        go.Bar(
            name=f"today = {today_label}",
            x=[f"yesterday = {y}" for y in subset["yesterday"]],
            y=(subset["P(wet tomorrow | today AND yesterday)"] * 100).tolist(),
            hovertemplate="%{x}<br>P(wet tomorrow)=%{y:.1f}%<extra></extra>",
        )
    )
    fig.add_hline(
        y=subset["P(wet tomorrow | today only)"].iloc[0] * 100,
        line=dict(dash="dash", color="crimson", width=1.5),
        annotation_text=f"Markov forecast for today={today_label}: {subset['P(wet tomorrow | today only)'].iloc[0]:.0%}",
        annotation_position="top left" if today_state == 0 else "bottom left",
    )
fig.update_layout(
    title="If yesterday truly didn't matter, every bar would sit on its dashed line",
    yaxis_title="P(wet tomorrow), %",
    barmode="group",
)
show(fig, "memory-test", height=460)

### Mostly not, with one exception

When today is **wet**, tomorrow's forecast barely moves whether yesterday was wet (68%) or dry (66%); the Markov assumption holds. When today is **dry**, yesterday matters: if yesterday was *also* dry, there is an 81% chance of staying dry tomorrow; if yesterday was *wet*, that drops to 58%. A dry day right after rain is less "settled" than a dry day in the middle of a dry spell.

So the memoryless assumption is not exactly true. Yesterday leaves a real, measurable echo, especially at the tail end of a wet spell. But it is a second-order effect: for a first pass at day-to-day persistence, "today" does most of the work.


### What the chain gets right

Even an imperfect chain is useful if it reproduces the right long-run behaviour. We simulate 40 independent weather histories from the fitted two-state chain and watch the running fraction of wet days settle down, the same "many plausible futures" idea as the ensemble forecasts in the previous module, with a far simpler engine.


In [9]:
rng = np.random.default_rng(11)
n_chains, n_days = 40, 150
start_state = 0  # start dry

running_wet_frac = np.zeros((n_chains, n_days))
for c in range(n_chains):
    state = start_state
    wet_count = 0
    for d in range(n_days):
        state = rng.choice(2, p=trans_prob[state])
        wet_count += state
        running_wet_frac[c, d] = wet_count / (d + 1)

In [10]:
fig = go.Figure()
days = list(range(1, n_days + 1))
for c in range(n_chains):
    fig.add_trace(
        go.Scatter(
            x=days,
            y=running_wet_frac[c].tolist(),
            mode="lines",
            line=dict(color="#1b6ca8", width=1),
            opacity=0.25,
            hoverinfo="skip",
            showlegend=False,
        )
    )
fig.add_trace(
    go.Scatter(
        x=days,
        y=running_wet_frac.mean(axis=0).tolist(),
        mode="lines",
        line=dict(color="crimson", width=2.5),
        name="Average across simulations",
    )
)
fig.add_hline(
    y=float(observed[1]),
    line=dict(dash="dash", color="black", width=1.5),
    annotation_text=f"Actually observed in Seattle: {observed[1]:.1%} of days",
)
fig.update_layout(
    title="40 simulated weather histories, all starting dry",
    xaxis_title="Day of simulation",
    yaxis_title="Running fraction of wet days",
    legend=dict(orientation="h", y=1.12),
)
show(fig, "mc-convergence", height=460)

Every simulated run wobbles for the first couple of weeks, which is luck in a short run, but they all settle around the wet fraction observed in the real data. This is the payoff: a slightly wrong, memoryless model can still nail the **climatology** (the long-run statistics) while remaining genuinely uncertain about any individual day. It is the same lesson as the ensemble forecasts, at a fraction of the computational cost.


### Why "good enough" works here

Real weather systems (fronts, high- and low-pressure systems, atmospheric rivers) move and evolve over days, not hours. So **today's weather already encodes most of the recent history**: if it is raining today, a system has probably been overhead for a day or two already, whether or not the model was told. A first-order chain is not ignoring memory so much as absorbing it into "today's state".

In terms of the previous module, a Markov chain does not fight the atmosphere's chaos. It harvests the short (1–3 day) stretch of genuine persistence that survives before chaos erases it, using the cheapest model that can do so.


### Where it breaks down

The memory test already caught the crack: dry days after a wet spell behave differently from dry days deep in a dry spell, and a first-order chain misses that. Two common fixes:

- **Higher-order Markov chains** condition on the last *k* days instead of one, trading simplicity for a little accuracy and needing far more data to estimate reliably.
- **Slower, non-Markov memory**: the real long-range predictability in weather comes not from yesterday but from boundary conditions that persist for months, such as ocean temperatures, El Niño/La Niña and soil moisture. That is the seasonal-outlook machinery from the previous module, on a timescale no short-lag Markov chain can see.


## Key takeaways

- A **Markov chain** assumes tomorrow depends only on today, a bold "memoryless" simplification.
- Fitted to four years of Seattle weather, the two-state chain is **internally consistent**: its stationary distribution matches the observed climate.
- The memoryless assumption is **mostly but not perfectly true**: yesterday leaves a measurable fingerprint, especially where a wet spell tapers into dry days.
- Simulated chains still reproduce the right **long-run statistics**, the same payoff as a full ensemble forecast at a tiny fraction of the cost.
- It works because weather systems persist for a few days on their own: the chain cheaply captures the window of persistence that chaos has not yet erased.


<!-- CURRICULUM_FOOTER_START -->
<div class="mf"><a class="mf-prev" href="/blogs/rainfall-predictability/why-cant-we-forecast-rain-six-months-out.html"><span>Previous</span>Why can&#x27;t we forecast rain six months out?</a><a class="mf-up" href="/series/prediction/index.html"><span>Series</span>Weather, Climate and Prediction</a><a class="mf-next" href="/blogs/detrending/detrend_hk_temperature.html"><span>Next</span>Decomposing Hong Kong temperature</a></div>
<!-- CURRICULUM_FOOTER_END -->